# 📚 Book Store Data Analysis Project

Hi! This is a project I built to practice data analyst skills.

I scraped book data from a practice website, cleaned it up, stored it in MySQL, ran some SQL queries, and built a Tableau dashboard.

**Tools I used:**
- Python + Selenium (web scraping)
- pandas (data cleaning + analysis)
- MySQL (storing data + SQL queries)
- Tableau (dashboard)

**Questions I wanted to answer:**
- Do expensive books actually get better ratings?
- Which books are the best value for money?
- How many books are in stock vs out of stock?
- Which expensive books have bad ratings?


## Step 0 — Install libraries and create the data folder

In [1]:
%pip install selenium pandas mysql-connector-python sqlalchemy pymysql

  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.20
    Uninstalling urllib3-1.26.20:
      Successfully uninstalled urllib3-1.26.20
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyppeteer 2.0.0 requires urllib3<2.0.0,>=1.25.8, but you have urllib3 2.5.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

os.makedirs("data", exist_ok=True)
print("All set!")

All set!


---
## Step 1 — Scraping the Data with Selenium

I'm scraping book titles, prices, star ratings and stock status from **books.toscrape.com**.
This site is made for practice so it's fine to scrape it.

I'm going through 5 pages which gives me about 100 books.


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# run Chrome in the background so no window pops up
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
print("Chrome started!")

Chrome started!


In [4]:
all_books = []

for page in range(1, 6):
    url = f"http://books.toscrape.com/catalogue/page-{page}.html"
    driver.get(url)
    time.sleep(2)  # wait for the page to load

    books = driver.find_elements(By.CLASS_NAME, "product_pod")

    for book in books:
        title  = book.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
        price  = book.find_element(By.CLASS_NAME, "price_color").text
        rating = book.find_element(By.CLASS_NAME, "star-rating").get_attribute("class")
        stock  = book.find_element(By.CLASS_NAME, "availability").text.strip()

        all_books.append({"title": title, "price": price, "rating": rating, "availability": stock})

    print(f"Page {page} done — {len(all_books)} books collected so far")

driver.quit()
print(f"\nDone! Scraped {len(all_books)} books total")

Page 1 done — 20 books collected so far
Page 2 done — 40 books collected so far
Page 3 done — 60 books collected so far
Page 4 done — 80 books collected so far
Page 5 done — 100 books collected so far

Done! Scraped 100 books total


In [5]:
df_raw = pd.DataFrame(all_books)
df_raw.to_csv("data/raw_books.csv", index=False)

print("Raw data saved to data/raw_books.csv")
print(f"Shape: {df_raw.shape}")
display(df_raw.head())

Raw data saved to data/raw_books.csv
Shape: (100, 4)


,title,price,rating,availability
0,A Light in the Attic,£51.77,star-rating Three,In stock
1,Tipping the Velvet,£53.74,star-rating One,In stock
2,Soumission,£50.10,star-rating One,In stock
3,Sharp Objects,£47.82,star-rating Four,In stock
4,Sapiens: A Brief History of Humankind,£54.23,star-rating Five,In stock


In [6]:
print("Sample price values: ",  df_raw["price"].unique()[:5])
print("Sample rating values:", df_raw["rating"].unique()[:5])
print("Sample availability: ",  df_raw["availability"].unique())

Sample price values:  ['£51.77' '£53.74' '£50.10' '£47.82' '£54.23']
Sample rating values: ['star-rating Three' 'star-rating One' 'star-rating Four'
 'star-rating Five' 'star-rating Two']
Sample availability:  ['In stock']


---
## Step 2 — Cleaning the Data with pandas

The raw data needs some fixing:
- The **price** column has a £ sign so Python thinks it's text, not a number
- The **rating** is a CSS class like `star-rating Three` instead of just `3`
- I want to add a **price category** column to group books as Cheap / Medium / Expensive


In [7]:
df = pd.read_csv("data/raw_books.csv")

# fix the price — strip the £ sign and convert to float
df["price"] = df["price"].str.replace("£", "").str.replace("Â", "").astype(float)

print("Price column fixed!")
print(df["price"].head(5))

Price column fixed!
0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: price, dtype: float64


In [8]:
# fix the rating — map the CSS class word to a number
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df["rating"] = df["rating"].str.replace("star-rating ", "")
df["rating"] = df["rating"].map(rating_map)

print("Rating column fixed!")
print(df["rating"].value_counts().sort_index())

Rating column fixed!
rating
1    22
2    19
3    22
4    18
5    19
Name: count, dtype: int64


In [9]:
# fix the availability column — just Yes or No
df["in_stock"] = df["availability"].apply(lambda x: "Yes" if "In stock" in x else "No")
df.drop(columns=["availability"], inplace=True)

print("In stock counts:")
print(df["in_stock"].value_counts())

In stock counts:
in_stock
Yes    100
Name: count, dtype: int64


In [10]:
# add a price category column
def price_category(price):
    if price < 15:
        return "Cheap"
    elif price < 35:
        return "Medium"
    else:
        return "Expensive"

df["price_category"] = df["price"].apply(price_category)

print("Price categories:")
print(df["price_category"].value_counts())

Price categories:
price_category
Expensive    50
Medium       41
Cheap         9
Name: count, dtype: int64


In [11]:
print("Missing values:")
print(df.isnull().sum())

df.to_csv("data/clean_books.csv", index=False)
print("\nClean data saved to data/clean_books.csv!")
display(df.head(10))

Missing values:
title             0
price             0
rating            0
in_stock          0
price_category    0
dtype: int64

Clean data saved to data/clean_books.csv!


,title,price,rating,in_stock,price_category
0,A Light in the Attic,51.77,3,Yes,Expensive
1,Tipping the Velvet,53.74,1,Yes,Expensive
2,Soumission,50.10,1,Yes,Expensive
3,Sharp Objects,47.82,4,Yes,Expensive
4,Sapiens: A Brief History of Humankind,54.23,5,Yes,Expensive
5,The Requiem Red,22.65,1,Yes,Medium
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,Yes,Medium
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,Yes,Medium
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,Yes,Medium
9,The Black Maria,52.15,1,Yes,Expensive


---
## Step 3 — Exploring the Data with pandas

Before loading into MySQL I want to do some quick analysis to understand the data.


In [12]:
print("=" * 45)
print("BASIC STATS")
print("=" * 45)
print(f"Total books:     {len(df)}")
print(f"Average price:   £{df['price'].mean():.2f}")
print(f"Cheapest book:   £{df['price'].min():.2f}")
print(f"Most expensive:  £{df['price'].max():.2f}")
print(f"Average rating:  {df['rating'].mean():.1f} out of 5")

BASIC STATS
Total books:     100
Average price:   £34.56
Cheapest book:   £10.16
Most expensive:  £58.11
Average rating:  2.9 out of 5


In [13]:
# does paying more actually get you a better book?
print("Average price per star rating:")
display(df.groupby("rating")["price"].mean().round(2).rename("avg_price"))

Average price per star rating:


rating
1    35.52
2    35.91
3    36.84
4    33.98
5    30.01
Name: avg_price, dtype: float64

In [14]:
print("Average rating per price category:")
display(df.groupby("price_category")["rating"].mean().round(2).rename("avg_rating"))

Average rating per price category:


price_category
Cheap        3.22
Expensive    2.76
Medium       3.07
Name: avg_rating, dtype: float64

In [15]:
print("Top 10 best value books (5 stars, cheapest first):")
best_value = df[df["rating"] == 5].sort_values("price").head(10)
display(best_value[["title", "price", "rating", "price_category"]])

Top 10 best value books (5 stars, cheapest first):


,title,price,rating,price_category
81,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Cheap
80,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Cheap
34,Sophie's World,15.94,5,Medium
54,Thirst,17.27,5,Medium
12,Set Me Free,17.46,5,Medium
30,The Four Agreements: A Practical Guide to Pers...,17.66,5,Medium
65,The Inefficiency Assassin: Time Management Tac...,20.59,5,Medium
43,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5,Medium
32,The Elephant Tree,23.82,5,Medium
23,Chase Me (Paris Nights #2),25.27,5,Medium


In [16]:
summary = df.groupby("price_category").agg(
    total_books    = ("title",    "count"),
    avg_price      = ("price",    "mean"),
    avg_rating     = ("rating",   "mean"),
    in_stock_count = ("in_stock", lambda x: (x == "Yes").sum())
).round(2).reset_index()

print("Summary by price category:")
display(summary)

summary.to_csv("data/summary.csv", index=False)
print("\nSummary saved!")

Summary by price category:


,price_category,total_books,avg_price,avg_rating,in_stock_count
0,Cheap,9,13.46,3.22,9
1,Expensive,50,47.48,2.76,50
2,Medium,41,23.44,3.07,41



Summary saved!


---
## Step 4 — Loading Data into MySQL

Now I'm putting the data into a MySQL database so I can practice writing SQL queries.

> ⚠️ **Before running this:** change `your_password` to your actual MySQL root password and make sure MySQL is running.


In [17]:
import mysql.connector

# ✏️ change these to your MySQL credentials
DB_USER     = "root"
DB_PASSWORD = "1234"
DB_HOST     = "localhost"
DB_NAME     = "bookstore"

# connect without a database first so we can create it
conn = mysql.connector.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD)
cursor = conn.cursor()

cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME};")
cursor.execute(f"USE {DB_NAME};")
print(f"Database '{DB_NAME}' ready!")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id             INT AUTO_INCREMENT PRIMARY KEY,
        title          VARCHAR(500),
        price          DECIMAL(6, 2),
        rating         INT,
        in_stock       VARCHAR(5),
        price_category VARCHAR(20)
    );
""")

# clear old data so this cell is safe to re-run
cursor.execute("TRUNCATE TABLE books;")
conn.commit()
print("Table ready!")

Database 'bookstore' ready!
Table ready!


In [18]:
df_load = pd.read_csv("data/clean_books.csv")

for _, row in df_load.iterrows():
    cursor.execute("""
        INSERT INTO books (title, price, rating, in_stock, price_category)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        row["title"],
        float(row["price"]),
        int(row["rating"]),
        row["in_stock"],
        row["price_category"]
    ))

conn.commit()
print(f"Loaded {len(df_load)} rows into MySQL!")

cursor.execute("SELECT COUNT(*) FROM books;")
print(f"Total rows in database: {cursor.fetchone()[0]}")

cursor.close()
conn.close()

Loaded 100 rows into MySQL!
Total rows in database: 100


---
## Step 5 — SQL Queries

Now the fun part — answering business questions with SQL!

I'm using **SQLAlchemy** to connect so pandas doesn't show a warning.
The queries themselves are exactly the same as normal SQL.


In [19]:
# SQLAlchemy engine — pandas works cleanly with this (no warnings!)
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

# helper so I don't have to repeat the connect code every time
def run_sql(query):
    with engine.connect() as con:
        return pd.read_sql(text(query), con)

print("Connected! Ready to run queries.")

Connected! Ready to run queries.


In [20]:
# Q1: how many books are in each price category?
q1 = run_sql("""
    SELECT price_category, COUNT(*) AS total_books
    FROM books
    GROUP BY price_category
    ORDER BY total_books DESC
""")
print("Q1 — Books per price category:")
display(q1)

Q1 — Books per price category:


,price_category,total_books
0,Expensive,50
1,Medium,41
2,Cheap,9


In [21]:
# Q2: average price per star rating
# does paying more actually mean a better book?
q2 = run_sql("""
    SELECT
        rating AS stars,
        COUNT(*) AS num_books,
        ROUND(AVG(price), 2) AS avg_price
    FROM books
    GROUP BY rating
    ORDER BY rating DESC
""")
print("Q2 — Average price per star rating:")
display(q2)

Q2 — Average price per star rating:


,stars,num_books,avg_price
0,5,19,30.01
1,4,18,33.98
2,3,22,36.84
3,2,19,35.91
4,1,22,35.52


In [22]:
# Q3: best value books — 5 stars and cheapest price
q3 = run_sql("""
    SELECT title, price, rating, price_category
    FROM books
    WHERE rating = 5
    ORDER BY price ASC
    LIMIT 10
""")
print("Q3 — Best value 5-star books:")
display(q3)

Q3 — Best value 5-star books:


,title,price,rating,price_category
0,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Cheap
1,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Cheap
2,Sophie's World,15.94,5,Medium
3,Thirst,17.27,5,Medium
4,Set Me Free,17.46,5,Medium
5,The Four Agreements: A Practical Guide to Pers...,17.66,5,Medium
6,The Inefficiency Assassin: Time Management Tac...,20.59,5,Medium
7,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5,Medium
8,The Elephant Tree,23.82,5,Medium
9,Chase Me (Paris Nights #2),25.27,5,Medium


In [23]:
# Q4: do expensive books get better ratings?
q4 = run_sql("""
    SELECT
        price_category,
        ROUND(AVG(rating), 2) AS avg_rating,
        COUNT(*) AS total_books
    FROM books
    GROUP BY price_category
    ORDER BY avg_rating DESC
""")
print("Q4 — Average rating per price category:")
display(q4)

Q4 — Average rating per price category:


,price_category,avg_rating,total_books
0,Cheap,3.22,9
1,Medium,3.07,41
2,Expensive,2.76,50


In [24]:
# Q5: how many books are in stock vs out of stock?
q5 = run_sql("""
    SELECT in_stock, COUNT(*) AS total
    FROM books
    GROUP BY in_stock
""")
print("Q5 — Stock availability:")
display(q5)

Q5 — Stock availability:


,in_stock,total
0,Yes,100


In [25]:
# Q6: expensive books with bad ratings — possible markdown candidates
q6 = run_sql("""
    SELECT title, price, rating
    FROM books
    WHERE price > 40 AND rating <= 2
    ORDER BY price DESC
""")
print("Q6 — Expensive books with bad ratings:")
display(q6)

Q6 — Expensive books with bad ratings:


,title,price,rating
0,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
1,Masks and Shadows,56.40,2
2,The Secret of Dreadwillow Carse,56.13,1
3,The Electric Pencil: Drawings from Inside Stat...,56.06,1
4,Judo: Seven Steps to Black Belt (an Introducto...,53.90,2
5,Tipping the Velvet,53.74,1
6,The Black Maria,52.15,1
7,Libertarianism for Beginners,51.33,2
8,"Saga, Volume 5 (Saga (Collected Editions) #5)",51.04,2
9,Soumission,50.10,1


In [26]:
# Q7: full summary table
q7 = run_sql("""
    SELECT
        price_category,
        COUNT(*) AS total_books,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(AVG(rating), 2) AS avg_rating,
        SUM(CASE WHEN in_stock = 'Yes' THEN 1 ELSE 0 END) AS in_stock_count
    FROM books
    GROUP BY price_category
""")
print("Q7 — Full summary by price category:")
display(q7)

Q7 — Full summary by price category:


,price_category,total_books,avg_price,avg_rating,in_stock_count
0,Expensive,50,47.48,2.76,50.0
1,Medium,41,23.44,3.07,41.0
2,Cheap,9,13.46,3.22,9.0


---
## Step 6 — Export for Tableau

Saving CSV files so I can import them into Tableau to build a dashboard.

> **Tip:** You can also skip this and connect Tableau straight to MySQL:
> Go to **Connect → To a Server → MySQL** and enter `localhost`, your username, password, then select the `bookstore` database.


In [27]:
# export 1: all books
df_all = run_sql("SELECT * FROM books")
df_all.to_csv("data/tableau_all_books.csv", index=False)
print(f"Saved {len(df_all)} rows → data/tableau_all_books.csv")

# export 2: summary by category
df_sum = run_sql("""
    SELECT price_category,
           COUNT(*) AS total_books,
           ROUND(AVG(price), 2) AS avg_price,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM books
    GROUP BY price_category
""")
df_sum.to_csv("data/tableau_summary.csv", index=False)
print("Saved summary → data/tableau_summary.csv")

# export 3: rating breakdown
df_rat = run_sql("""
    SELECT rating AS stars,
           COUNT(*) AS total,
           ROUND(AVG(price), 2) AS avg_price
    FROM books
    GROUP BY rating
    ORDER BY rating DESC
""")
df_rat.to_csv("data/tableau_by_rating.csv", index=False)
print("Saved rating breakdown → data/tableau_by_rating.csv")

engine.dispose()
print("\nAll CSV files ready for Tableau!")

Saved 100 rows → data/tableau_all_books.csv
Saved summary → data/tableau_summary.csv
Saved rating breakdown → data/tableau_by_rating.csv

All CSV files ready for Tableau!


---
## All Done! 🎉

Here's everything that was created:

| File | What it is |
|------|------------|
| `data/raw_books.csv` | Raw scraped data from the website |
| `data/clean_books.csv` | Cleaned up version ready for analysis |
| `data/summary.csv` | Summary grouped by price category |
| `data/tableau_all_books.csv` | Full dataset for Tableau |
| `data/tableau_summary.csv` | Category summary for Tableau |
| `data/tableau_by_rating.csv` | Rating breakdown for Tableau |

---

### Tableau Dashboard Ideas
- Bar chart: number of books per price category
- Scatter plot: price vs rating (does price = quality?)
- KPI cards: total books, avg price, avg rating, % in stock
- Table: top 10 best value books

---

### What I Learned
- Selenium needs `time.sleep()` otherwise it scrapes before the page loads
- Data cleaning takes longer than expected — just the price column needed 3 fixes
- `GROUP BY` in SQL is really useful for summarizing data quickly
- Use SQLAlchemy with pandas instead of the raw connector to avoid warnings
- Connecting Tableau directly to MySQL is easier than exporting CSVs
